# RAG Embedding Layer
### Ein geführter Workshop zur zweiten Offline-Stufe: von persistierten Chunks zu prüfbaren Vektoren

---

## Anschluss an die Ingestion Layer

Die Ingestion Layer ist abgeschlossen. Ihr Ergebnis ist `chunks.jsonl`, eine Datei,
in der jede Zeile einen einzelnen Chunk als JSON-Objekt enthält. Jeder Chunk hat eine
eindeutige `id`, eine `document_id`, ein `text`-Feld und `metadata`. Wie diese Chunks
entstanden sind, also Cleaning, Chunking-Strategie und Fallback-Mechanismus, ist für dieses
Notebook irrelevant. Die Embedding Layer hat keinen Zugriff auf Rohdaten und braucht
ihn auch nicht.

Die Embedding Layer beginnt genau hier:

```
chunks.jsonl
  → Chunk laden
  → chunk["text"] extrahieren
  → Embedding-Modell
  → Vektor
  → speichern mit chunk_id + metadata
```

---

### Offline vs. Online: eine wichtige Unterscheidung

**Offline** bedeutet hier: vorab berechnet, einmalig (oder bei Datenänderung),
nicht pro Nutzerfrage.

**Online** bedeutet hier: zur Anfragezeit, also für jede einzelne Query.

Das hat nichts damit zu tun, ob ein Modell lokal oder über eine API läuft.
Die Chunk-Embeddings werden offline erzeugt und gespeichert. Der Query-Vektor
einer Nutzerfrage wird online erzeugt und sofort verworfen.

---

### Warum die Embedding Layer getrennt von der Ingestion ist

Die Trennung hat konkrete Konsequenzen:

**Modellwechsel ohne Re-Ingestion.** Wenn ein neues Embedding-Modell evaluiert werden
soll, muss nur die Embedding Layer neu laufen. Die Chunks bleiben unverändert.

**Vergleichbarkeit.** Dasselbe `chunks.jsonl` kann mit zwei verschiedenen Embedding-Modellen
verarbeitet werden. Die resultierenden Vektoren sind direkt vergleichbar, weil sie auf
identischen Chunks basieren.

**Keine Rohdatenabhängigkeit.** Die Embedding Layer weiß nicht, ob die Quelle Markdown
oder JSONL war. Sie liest nur strukturierte Chunks.

**Reproduzierbarkeit.** Gleiche Chunks + gleiches Modell = identische Vektoren.

---

## 1. Input der Embedding Layer: persistierte Chunks

Die Embedding Layer hat genau einen Input: `chunks.jsonl`. Kein Loader, kein Cleaner,
kein Chunker wird aufgerufen.

In [ ]:
from pathlib import Path
from rag.ingestion.storage.chunk_loader import load_chunks

chunk_store_path = Path("/home/jovyan/workspace/data/processed/chunks.jsonl")

print(f"Chunk-Store : {chunk_store_path}")
print(f"Existiert   : {chunk_store_path.exists()}")

if chunk_store_path.exists():
    size_kb = chunk_store_path.stat().st_size / 1024
    print(f"Dateigroesse  : {size_kb:.1f} KB")

In [ ]:
# load_chunks() liest chunks.jsonl streaming - kein vollständiges RAM-Laden nötig.
# Für die Inspektion materialisieren wir die Liste einmalig.
chunks = list(load_chunks(chunk_store_path))

print(f"Chunks geladen: {len(chunks)}")
print()

In [ ]:
if not chunks:
    raise RuntimeError("Keine Chunks gefunden. Bitte zuerst das Ingestion-Notebook ausführen.")

example = chunks[0]

print("Struktur eines Chunks:")
print()
print(f"  id          : {example['id']}")
print(f"  document_id : {example['document_id']}")
print(f"  text[:120]  : {repr(example['text'][:120])}")
print(f"  metadata    :")
for k, v in example["metadata"].items():
    print(f"    {k:<25}: {v}")

**Warum genau diese vier Felder wichtig sind:**

| Feld | Rolle in der Embedding Layer |
|------|------------------------------|
| `id` | Verbindet den gespeicherten Vektor zurück mit dem Chunk. Ohne diese Verbindung wäre Retrieval blind. |
| `document_id` | Erlaubt späteren Systemen, alle Vektoren eines Dokuments zu finden, etwa für gezieltes Re-Embedding bei Dokumentänderungen. |
| `text` | Das und ausschließlich das geht ins Embedding-Modell. |
| `metadata` | Wird unverändert in den Embedding-Record übernommen. Retrieval-Systeme können damit filtern oder Ergebnisse erklären. |

Der `text` ist der einzige Teil, der die Systemgrenze zur Embedding-Berechnung passiert.
Alles andere ist begleitende Verwaltungsinformation, allerdings notwendige.

---

## 2. Embedding-Bereitschaft prüfen

Embedding-Berechnungen sind teuer, denn jeder Chunk erfordert einen Modell-Forward-Pass.
Fehler, die erst nach der Hälfte der Chunks auftreten, kosten Zeit und hinterlassen
einen inkompletten Embedding-Bestand.

Deshalb: Validierung **vor** dem ersten Modellaufruf. Das zugrunde liegende Prinzip: Fehler sollen sofort sichtbar werden, nicht still ignoriert werden.

In [ ]:
ids = [c["id"] for c in chunks]
n   = len(chunks)

checks = {
    "chunks.jsonl vorhanden"       : chunk_store_path.exists(),
    "Mindestens 1 Chunk"           : n > 0,
    "Feld 'text' vorhanden"        : all("text" in c for c in chunks),
    "text nicht-leer"              : all(isinstance(c.get("text"), str) and c["text"].strip() for c in chunks),
    "Feld 'id' vorhanden"          : all("id" in c and c["id"] for c in chunks),
    "IDs eindeutig"                : len(set(ids)) == n,
    "Feld 'document_id' vorhanden" : all("document_id" in c and c["document_id"] for c in chunks),
    "Feld 'metadata' vorhanden"    : all(isinstance(c.get("metadata"), dict) for c in chunks),
}

all_ok = all(checks.values())

print("Embedding-Readiness-Check:")
print()
for label, result in checks.items():
    status = "OK  " if result else "FAIL"
    print(f"  [{status}]  {label}")

print()
if not all_ok:
    raise RuntimeError("Embedding-Readiness-Check fehlgeschlagen - siehe FAIL oben.")

print(f"EMBEDDING-READY  ({n} Chunks)")

**Warum leere Texte blockiert werden:**

Ein leerer Text erzeugt einen numerisch validen Vektor, der aber semantisch nichts repräsentiert. Er landet im Vektorindex und verfälscht Retrieval-Ergebnisse.
Wäre die Validierung nicht vorhanden, würde dieser Fehler erst deutlich später auffallen:
bei schlechten Retrieval-Ergebnissen, die sich nicht erklären lassen.

**Warum doppelte IDs blockiert werden:**

Jede ID wird später der Schlüssel eines Eintrags im Vektorindex. Doppelte IDs bedeuten
entweder Überschreibung oder Inkonsistenz. Beide Konsequenzen sind schwer zu diagnostizieren,
weil sie erst bei Retrieval sichtbar werden.

---

## 3. Was ist ein Embedding im Systemkontext?

Ein Embedding ist ein numerischer Vektor fixer Länge, der die semantische Bedeutung
eines Textes im Vektorraum repräsentiert.

Ein Embedding-Modell nimmt einen String entgegen und gibt eine Liste von
Fließkommazahlen zurück, etwa `[0.023, -0.412, 0.887, ...]`, mit typischerweise
256 bis 4096 Elementen. Diese Zahlen einzeln zu lesen ergibt keinen Sinn.
Sie sind nicht interpretierbar wie Tokens oder Keywords.

Was sie bedeuten, entsteht erst durch ihre **Position im Raum relativ zu anderen
Vektoren**: Texte mit ähnlicher semantischer Bedeutung landen nah beieinander,
unabhängig davon, ob sie dieselben Wörter verwenden.

```
"Wie funktioniert der Attention-Mechanismus?"
    → [0.23, -0.11, 0.87, ...]   (Query-Vektor)

"Der Attention-Mechanismus gewichtet relevante Tokens."
    → [0.25, -0.09, 0.84, ...]   (naher Chunk-Vektor)

"Das Wetter in München ist heute wolkig."
    → [-0.54, 0.33, -0.12, ...]  (weit entfernter Chunk-Vektor)
```

Retrieval nutzt genau diese Eigenschaft: Ein Query-Text wird ebenfalls eingebettet,
und der Vektorindex sucht die Chunk-Vektoren, die dem Query-Vektor am nächsten liegen.

---

### Warum Dimension stabil sein muss

Alle Vektoren im Index müssen dieselbe Dimension haben. Das ist eine mathematische
Voraussetzung jedes Ähnlichkeitsvergleichs. Kein Vektorindex der Welt kann einen
768-dim-Vektor mit einem 1024-dim-Vektor vergleichen.

Das hat eine direkte Konsequenz: **Modellwechsel bedeutet Index-Neubau.** Wenn das
Embedding-Modell gewechselt wird, sind alle gespeicherten Vektoren mit dem neuen
Modell inkompatibel und müssen neu berechnet werden.

---

### Warum Query- und Dokument-Embedding getrennt sind

Manche Embedding-Modelle behandeln Queries und Dokumente unterschiedlich. Ein Query
ist eine kurze Frage oder Phrase. Ein Dokument-Chunk ist ein Textabschnitt.
Einige Modelle verwenden unterschiedliche Formatierungen oder Präfixe, um dem Modell
diese Rollentrennung zu signalisieren.

In der Ingestion-Pipeline werden Chunks als **Dokumente** eingebettet.
Query-Embedding passiert online, also zur Anfragezeit, für jede Query separat.
Das ist der Grund, warum `embed_documents()` und `embed_queries()` im
Embedder-Contract getrennt existieren.

---

## 4. Embedding-Modell konfigurieren

### 4.1 Der BaseEmbedder-Contract

Die Embedding Layer arbeitet nicht direkt mit konkreten Modellen. Sie arbeitet gegen
einen stabilen Contract: die abstrakte Klasse `BaseEmbedder`.

Dieser Contract definiert, was jedes Embedding-Modell können **muss**:

```python
embed_documents(texts: List[str]) -> List[List[float]]
embed_queries(texts: List[str])   -> List[List[float]]
dimension()                       -> Optional[int]
supported_modes()                 -> Set[str]
```

Der `EmbeddingOrchestrator` muss sich darauf verlassen können, dass jedes Modell
diese Fähigkeiten hat, unabhängig davon, ob es sich um BGE-M3, Gemma oder ein
zukünftiges Modell handelt.

---

### 4.2 Dense, Sparse, Hybrid: warum der Retrieval-Modus Teil der Architektur ist

**Dense:** Das klassische Vektorembedding. Ein Text wird in einen Float-Vektor
übersetzt. Ähnlichkeit wird über Vektordistanz gemessen (z.B. Cosine Similarity).
Stärke: semantische Suche, die Bedeutung findet und nicht nur Keywords.

**Sparse:** Lexikalische Gewichtungen. Jedes relevante Token bekommt ein Gewicht
(ähnlich wie BM25). Stärke: präzise Keyword-Suche, gut bei seltenen Fachbegriffen.

**Hybrid:** Beide gleichzeitig. Ein einziger Forward-Pass erzeugt Dense- und
Sparse-Repräsentation. Stärke: kombiniert semantische Suche mit lexikalischer Präzision.

Derselbe Chunk kann unterschiedliche Repräsentationen haben, je nach Retrieval-Modus.
BGE-M3 unterstützt alle drei Modi. Gemma unterstützt nur Dense. Der Retrieval-Modus
muss deshalb Teil der Konfiguration sein, denn er bestimmt, was gespeichert wird und
was der Vektorindex später erwartet.

---

### 4.3 Der ModelCache: Modelle als teure Runtime-Ressourcen

Ein Embedding-Modell zu laden bedeutet: Gewichte aus dem Dateisystem lesen, auf GPU
oder CPU übertragen, Inferenz-Strukturen initialisieren. Das dauert Sekunden bis
Minuten und belegt GPU-Speicher.

Der `ModelCache` verhindert doppeltes Laden: Er hält Modell-Objekte im Prozess-Speicher
und gibt bei erneutem `get_or_load()` dieselbe Instanz zurück. Der Cache arbeitet
pro Prozess, ist thread-safe und LRU-begrenzt.

In [ ]:
from rag.embedding.config import (
    EmbeddingConfig,
    EmbeddingBehaviorConfig,
    EmbeddingProjectionConfig,
)
from rag.embedding.factory import create_embedder
from rag.embedding.model_cache import get_default_cache

# Konfiguration für BGE-M3 im Dense-Modus (CPU, kein fp16 für Demo)
config = EmbeddingConfig(
    provider="bge-m3",
    model_name="BAAI/bge-m3",
    model_version="1.0",
    device="cpu",
    batch_size=16,
    use_fp16=False,
    retrieval_mode="dense",
    behavior=EmbeddingBehaviorConfig(
        normalize=True,
        mode="document",
    ),
    projection=EmbeddingProjectionConfig(
        target_dim=None,   # keine Projektion
        method="truncate",
        model_aware=True,
    ),
)

print("EmbeddingConfig:")
print(f"  provider        : {config.provider}")
print(f"  model_name      : {config.model_name}")
print(f"  model_version   : {config.model_version}")
print(f"  device          : {config.device}")
print(f"  batch_size      : {config.batch_size}")
print(f"  retrieval_mode  : {config.retrieval_mode}")
print(f"  normalize       : {config.behavior.normalize}")
print(f"  projection      : target_dim={config.projection.target_dim}")

### Kleine Experimentideen zu dieser Konfiguration

- `retrieval_mode="hybrid"` statt `"dense"`, da BGE-M3 beide unterstützt.
  Beobachte anschließend in Abschnitt 9, ob `sparse_embedding` im Record gefüllt ist.
- `batch_size=4` statt `16`. In Abschnitt 6 ist der Laufzeitunterschied messbar.
- `normalize=False` in `EmbeddingBehaviorConfig`. Der gespeicherte Vektor hat
  dann eine L2-Norm ≠ 1.0. In Abschnitt 7 wird sichtbar, was das bedeutet.

In [ ]:
# Modell laden
embedder = create_embedder(config, cache=get_default_cache())

print(f"Embedder geladen  : {embedder.model_type()}")
print(f"Dimension         : {embedder.dimension()}")
print(f"Unterstuetzte Modi : {sorted(embedder.supported_modes())}")
print(f"Standard-Modus    : {embedder.default_retrieval_mode()}")
print(f"MRL-faehig        : {embedder.is_mrl_model()}")
print()
print("Cache-Zustand:")
for key in get_default_cache().cached_keys():
    print(f"  {key}")

**Was dieser Output zeigt:**

Die Dimension ist die native Ausgabegröße des Modells, hier 1024 für BGE-M3.
Alle Vektoren, die dieses Modell erzeugt, haben exakt diese Länge. Ein Vektorindex,
der mit diesem Modell aufgebaut wird, erwartet 1024-dimensionale Vektoren
und weist andere zurück.

Der Cache-Eintrag `BAAI/bge-m3::cpu::float32` zeigt, dass das Modell jetzt im
Prozess-Cache liegt. Ein zweiter Aufruf von `create_embedder()` mit derselben
Konfiguration würde dasselbe Objekt aus dem Cache zurückgeben.

**Hinweis zu BGE-M3:** Das Modell unterstützt Dense, Sparse und Hybrid.
Wir verwenden hier Dense-Modus, weil es für die Didaktik klarer ist.
Im Hybrid-Modus wäre ein einziger Forward-Pass effizienter.

---

### 4.4 Alternativer Embedder: Gemma

Gemma (`google/embeddinggemma-300m`) unterscheidet sich in zwei wesentlichen Punkten
von BGE-M3:

**Instructional Prompts:** Gemma erwartet unterschiedliche Formatierungen für Queries
und Dokumente (`"task: search result | query: ..."` vs. `"title:  | text: ..."`).
BGE-M3 dagegen verwendet keine Präfixe.

**MRL-Support:** Gemma wurde mit Matryoshka Representation Learning trainiert.
Das bedeutet: `embed[:256]` ist nicht nur ein abgeschnittener Vektor, sondern
selbst eine valide, vollständige Repräsentation. BGE-M3 hat dieses Training nicht,
für BGE-M3 ist ein abgeschnittener Vektor Informationsverlust.

```python
# Gemma-Konfiguration (nur als Referenz, nicht ausführen)
gemma_config = EmbeddingConfig(
    provider="gemma",
    model_name="google/embeddinggemma-300m",
    model_version="1.0",
    device="cpu",
    use_bfloat16=False,    # bfloat16 korrekt für Gemma; float16 NICHT verwenden
    retrieval_mode="dense",
    projection=EmbeddingProjectionConfig(
        target_dim=256,    # MRL-valide Dimension für Gemma
        method="mrl",
        model_aware=True,
    ),
)
```

---

## 5. Einen einzelnen Chunk einbetten

Bevor alle Chunks verarbeitet werden, lohnt ein detaillierter Blick auf das,
was beim Einbetten eines einzelnen Chunks passiert.

In [ ]:
# Einen repräsentativen Chunk auswählen
sample_chunk = next(
    (c for c in chunks if len(c["text"]) > 100),
    chunks[0]
)

print("Ausgewählter Chunk:")
print(f"  id          : {sample_chunk['id']}")
print(f"  text-Laenge  : {len(sample_chunk['text'])} Zeichen")
print()
print("Text (erste 300 Zeichen):")
print(sample_chunk["text"][:300])

In [ ]:
# Embedding erzeugen - embed_documents() erwartet eine Liste von Strings
raw_vectors = embedder.embed_documents([sample_chunk["text"]])
vector = raw_vectors[0]

print("Embedding-Ergebnis:")
print(f"  Dimension       : {len(vector)}")
print(f"  Typ             : {type(vector[0]).__name__}")
print()
print("Erste 10 Werte:")
print(f"  {[round(x, 5) for x in vector[:10]]}")
print()
print("Letzte 5 Werte:")
print(f"  {[round(x, 5) for x in vector[-5:]]}")

### Kleine Experimentidee: embed_documents vs. embed_queries

- Rufe `embedder.embed_queries([sample_chunk["text"]])` auf und vergleiche die
  ersten 10 Werte mit dem obigen Ergebnis.
- Bei BGE-M3 sind die Werte identisch, das Modell unterscheidet Query und Dokument
  nicht beim Encoden.
- Für Gemma wären die Werte verschieden, weil unterschiedliche Präfixe injiziert werden.
- Die Trennung im Contract (`embed_documents` / `embed_queries`) ist also keine
  Redundanz, sondern eine modellspezifisch relevante Unterscheidung.

**Was dieser Vektor bedeutet und was er nicht bedeutet:**

Die einzelnen Zahlen sind nicht interpretierbar. Es gibt keine Dimension, die für
"Künstliche Intelligenz" steht. Die Bedeutung des Vektors liegt ausschließlich in
seiner **Position im 1024-dimensionalen Raum relativ zu anderen Vektoren**.

Was wir noch **nicht** sehen: Normalisierung und Projektion.
Diese Schritte passieren im Orchestrator, nicht direkt im Embedder.

In [ ]:
import math

# L2-Norm des rohen Vektors (vor Normalisierung durch den Orchestrator)
norm = math.sqrt(sum(x * x for x in vector))
print(f"L2-Norm des rohen Vektors: {norm:.6f}")
print()
print("Wenn die Norm ≠ 1.0 ist, ist der Vektor noch nicht normalisiert.")
print("Der Orchestrator normalisiert auf L2-Norm = 1.0, bevor er speichert.")

---

## 6. Der EmbeddingOrchestrator: Laufzeitsteuerung der Embedding Layer

Der `EmbeddingOrchestrator` ist die eigentliche Laufzeitsteuerung der gesamten
Embedding Layer.

**Was der Orchestrator macht und warum er zentral ist:**

- **Validierung:** Jeder Chunk wird vor der Verarbeitung geprüft. Fehler werden sofort geworfen.
- **Batching:** Chunks werden in Batches aufgeteilt, damit das Modell effizient arbeiten kann.
- **Retrieval-Modus-Steuerung:** Der Orchestrator entscheidet, ob `embed_documents()`,
  `embed_documents_sparse()` oder `embed_documents_hybrid()` aufgerufen wird.
- **Projektion:** Falls konfiguriert, wird der Vektor nach dem Einbetten projiziert.
- **Normalisierung:** L2-Normalisierung, falls konfiguriert.
- **Persistenz:** Falls `persist=True`, schreibt der Orchestrator batchweise in den Store.

Das Prinzip dahinter: Die Modelle selbst sollen nur encoden. Die gesamte
Pipeline-Steuerung liegt im Orchestrator.

### 6.1 Streaming-Architektur

Die Implementierung ist bewusst streaming-orientiert:

- `chunks: Iterable[dict]`: der Orchestrator erwartet keinen fertigen `List[dict]`.
  Er kann über einen Generator iterieren.
- `stream() -> Iterator[EmbeddingVector]`: Ergebnisse werden `yield`-weise zurückgegeben.
- Batchweises Schreiben: Der Store bekommt pro Batch eine Schreiboperation.

Das hat einen konkreten Vorteil: Bei 100.000 Chunks mit 1024-dim-Vektoren (Float32)
wäre das ca. 400 MB allein für die Vektoren, ohne Metadaten.
Streaming verhindert diese vollständige RAM-Akkumulation.

### 6.2 Batching: Warum Streaming + Batches kombiniert werden

Embedding-Modelle arbeiten effizienter auf Batches als auf Einzeltexten. Ein
GPU-Forward-Pass für 16 Texte ist schneller als 16 einzelne Forward-Pässe. Die Lösung
kombiniert beides: `batch_iter()` schneidet den Stream in Batches, ohne alle Elemente
gleichzeitig materialisieren zu müssen.

In [ ]:
from rag.embedding.batching import batch_iter

# batch_iter demonstrieren - arbeitet auf beliebigen Iterables
demo_items = list(range(10))

print("batch_iter Demo (10 Items, batch_size=3):")
print()
for i, batch in enumerate(batch_iter(demo_items, batch_size=3)):
    print(f"  Batch {i}: {batch}")

print()
print("Wichtig: batch_iter erwartet Iterable[T], nicht List[T].")
print("Es kann auf Generatoren, Datei-Readern, DB-Cursors arbeiten - keine len() noetig.")

In [ ]:
from rag.embedding.orchestrator import EmbeddingOrchestrator
import time

# Orchestrator ohne Store - zunächst ohne Persistenz
orchestrator = EmbeddingOrchestrator(
    embedder=embedder,
    config=config,
    store=None,
)

print("Orchestrator konfiguriert:")
print(f"  retrieval_mode    : {orchestrator._retrieval_mode}")
print(f"  projection_method : {orchestrator._projection_method}")
print(f"  batch_size        : {orchestrator.config.batch_size}")
print(f"  normalize         : {orchestrator.config.behavior.normalize}")

In [ ]:
# Alle Chunks einbetten - stream() ist der Memory-effiziente Pfad
print(f"Verarbeite {len(chunks)} Chunks mit batch_size={config.batch_size}...")
print()

t0 = time.time()
embedding_vectors = list(orchestrator.stream(iter(chunks), persist=False))
elapsed = time.time() - t0

print(f"Embeddings erzeugt : {len(embedding_vectors)}")
print(f"Zeit               : {elapsed:.1f}s")
print(f"Chunks/Sekunde     : {len(embedding_vectors)/elapsed:.1f}")

**Was dieser Output zeigt:**

Die zentrale Integritätsbedingung der Embedding Layer:
`chunks_loaded == embeddings_created`. Jeder Chunk hat genau einen Vektor.
Wenn diese Zahl nicht übereinstimmt, ist etwas schiefgelaufen, und das muss
vor der Persistenz bekannt sein.

---

## 7. Normalisierung und Projektion

### 7.1 L2-Normalisierung

Normalisierung bedeutet: Den Vektor so skalieren, dass seine L2-Norm (Euklidische Länge)
gleich 1 wird. Jede Zahl im Vektor wird durch die Norm geteilt.

Warum das sinnvoll ist: Bei Cosine Similarity, dem häufigsten Ähnlichkeitsmaß für
Vektoren, spielt die Länge des Vektors keine Rolle, nur die Richtung. Wenn Vektoren
vorab normalisiert sind, vereinfacht sich Cosine Similarity zu einem reinen
Skalarprodukt. Viele Vektorindizes (z.B. FAISS) nutzen das intern aus.

Wichtig: Normalisierung verändert **nicht** die semantische Bedeutung des Vektors.
Konsistenz ist entscheidend: Wenn manche Vektoren normalisiert und andere nicht
normalisiert sind, sind die Ähnlichkeitswerte nicht mehr vergleichbar.

In [ ]:
from rag.embedding.normalization import l2_normalize

# Vorher/Nachher für einen rohen Vektor
raw_vec  = embedder.embed_documents([sample_chunk["text"]])[0]
norm_vec = l2_normalize(raw_vec)

raw_norm   = math.sqrt(sum(x * x for x in raw_vec))
after_norm = math.sqrt(sum(x * x for x in norm_vec))

print("Normalisierung:")
print(f"  L2-Norm vorher  : {raw_norm:.6f}")
print(f"  L2-Norm nachher : {after_norm:.6f}   (≈ 1.0)")
print()
print("Erste 5 Werte - vorher vs. nachher:")
for i in range(5):
    print(f"  [{i}]  {raw_vec[i]:+.6f}  →  {norm_vec[i]:+.6f}")

### 7.2 Projektion: Dimension reduzieren, aber mit Bedacht

Manchmal ist die native Dimension eines Modells (z.B. 1024 für BGE-M3) größer als
nötig oder gewünscht. Kleinere Vektoren bedeuten: weniger Speicherbedarf im Index,
schnellere Ähnlichkeitssuche.

Die Implementierung unterstützt drei Projektionsmethoden:

**`truncate`:** Die ersten `target_dim` Werte des Vektors werden behalten, der Rest
wird abgeschnitten. Das ist eine **lossy** Operation. Es gibt keine semantische Garantie:
Die ersten 256 Dimensionen eines 1024-dim-Vektors repräsentieren nicht notwendigerweise
die "wichtigsten" Informationen.

**`mrl` (Matryoshka Representation Learning):** Das Modell wurde trainiert, Information
in verschachtelten Teilräumen zu kodieren. `vector[:256]` ist dann eine eigenständige,
vollständige Repräsentation, nicht weil sie abgeschnitten wurde, sondern weil das
Modell so trainiert wurde.

Der kritische Unterschied: `mrl ≠ truncate`. Truncation auf einem nicht-MRL-Modell
ist Informationsverlust. MRL-Projektion auf einem MRL-Modell ist semantisch valide.
BGE-M3 lehnt `mrl`-Projektion ab (kein MRL-Training), Gemma unterstützt sie.

**`pad`:** Vektoren werden auf eine größere Dimension mit Nullen aufgefüllt.

In [ ]:
from rag.embedding.projection import project_vector

# Projektion demonstrieren
full_vec = norm_vec  # normalisierter 1024-dim Vektor

truncated = project_vector(full_vec, target_dim=512, method="truncate")
padded    = project_vector(full_vec[:64], target_dim=128, method="pad")

print("Projektion Demo:")
print(f"  Ausgangs-Dimension        : {len(full_vec)}")
print(f"  Nach truncate auf 512     : {len(truncated)}")
print(f"  64-dim + pad auf 128      : {len(padded)}")
print()

# Normprüfung nach truncation
trunc_norm = math.sqrt(sum(x * x for x in truncated))
print(f"  L2-Norm nach truncate     : {trunc_norm:.6f}")
print("  (≠ 1.0 - truncation macht Re-Normalisierung nötig, falls Index das erwartet)")
print()
print("Hinweis: Der Orchestrator projiziert VOR der Normalisierung,")
print("damit der gespeicherte Vektor bereits normalisiert ist.")

---

## 8. Deterministische embedding_id: warum IDs nicht zufällig sein dürfen

Jeder gespeicherte Embedding-Datensatz braucht eine stabile, eindeutige ID.
Diese ID wird deterministisch aus drei Komponenten berechnet:

```
embedding_id = SHA-256(chunk_id | model_name | model_version)
```

Warum das architektonisch entscheidend ist:

**Reproduzierbarkeit.** Wenn dieselbe `chunk_id` mit demselben Modell und derselben
Version erneut eingebettet wird, entsteht dieselbe `embedding_id`. Der Vektorindex
kann den alten Eintrag zuverlässig überschreiben, ohne Duplikate zu erzeugen.

**Modellwechsel sind explizit.** Wenn `model_version` sich ändert, auch wenn der
Modellname gleich bleibt, ändert sich die `embedding_id`. Ein alter Eintrag im Index
bleibt erhalten und muss explizit gelöscht oder ersetzt werden. Das verhindert
**stille Inkonsistenzen**.

**Retrieval-Experimente bleiben konsistent.** Zwei Modellversionen sind eindeutig
unterscheidbar. Es gibt keine Verwechslung.

In [ ]:
from rag.embedding.utils import embedding_id

chunk_id = sample_chunk["id"]

# Determinismus: gleicher Input → gleiche ID, immer
id_v1    = embedding_id(chunk_id, "BAAI/bge-m3", "1.0")
id_v1b   = embedding_id(chunk_id, "BAAI/bge-m3", "1.0")  # identischer Aufruf
id_v2    = embedding_id(chunk_id, "BAAI/bge-m3", "2.0")  # Modell-Version gewechselt
id_other = embedding_id(chunk_id, "google/embeddinggemma-300m", "1.0")  # anderes Modell

print(f"chunk_id     : {chunk_id[:32]}...")
print()
print("embedding_id Varianten:")
print(f"  bge-m3 v1.0  (1. Aufruf) : {id_v1[:32]}...")
print(f"  bge-m3 v1.0  (2. Aufruf) : {id_v1b[:32]}...")
print(f"  bge-m3 v2.0              : {id_v2[:32]}...")
print(f"  embeddinggemma v1.0      : {id_other[:32]}...")
print()
print(f"  v1.0 == v1.0 (deterministisch): {id_v1 == id_v1b}")
print(f"  v1.0 == v2.0 (Modellwechsel)  : {id_v1 == id_v2}")

**Was der Output zeigt:**

Zwei identische Aufrufe erzeugen dieselbe ID, das ist deterministisches Verhalten.
Ein Versionswechsel von `1.0` auf `2.0` erzeugt eine völlig andere ID, das ist
beabsichtigtes Verhalten. Wenn das Modell neue Gewichte bekommt, sollen die alten
Embedding-IDs nicht wiederverwendet werden.

---

## 9. Aufbau eines Embedding-Records

Ein Embedding-Record ist nicht nur ein Float-Array. Er ist ein strukturierter Datensatz,
der den Vektor mit seinem Kontext verbindet. Ohne diesen Kontext wäre der Vektor
wertlos: Retrieval weiß nicht, woher er kommt, welches Modell ihn erzeugt hat,
oder welchen Chunk er repräsentiert.

In [ ]:
# Beispiel-Record ansehen
if not embedding_vectors:
    raise RuntimeError("Keine Embedding-Vektoren erzeugt - Zelle 6c zuerst ausführen.")

rec = embedding_vectors[0]

print("EmbeddingVector - vollständige Struktur:")
print()
print(f"  id                : {rec['id'][:32]}...  (SHA-256 aus chunk_id + model)")
print(f"  chunk_id          : {rec['chunk_id'][:32]}...")
print(f"  document_id       : {rec['document_id']}")
print(f"  embedding_type    : {rec['embedding_type']}")
print(f"  model_type        : {rec['model_type']}")
print(f"  projection_method : {rec['projection_method']}")
print(f"  embedding dim.    : {len(rec['embedding']) if rec['embedding'] else 'None'}")
print(f"  sparse_embedding  : {rec['sparse_embedding']}")
print(f"  metadata keys     : {list(rec['metadata'].keys())}")

**Warum jedes Feld existiert:**

| Feld | Warum es da ist |
|------|-----------------:|
| `id` | Primärschlüssel im Vektorindex. Deterministisch, erlaubt gezieltes Überschreiben. |
| `chunk_id` | Verbindung zurück zum Chunk in `chunks.jsonl`. Retrieval findet den Vektor, `chunk_id` holt den Text. |
| `document_id` | Erlaubt: "Alle Vektoren für Dokument X finden und löschen", für gezielte Aktualisierung. |
| `embedding_type` | War der Vektor dense, sparse oder hybrid? Retrieval muss das wissen. |
| `model_type` | Welches Modell hat den Vektor erzeugt? Notwendig für Diagnose und Reproduzierbarkeit. |
| `projection_method` | Wurde der Vektor projiziert? Falls ja: welche Methode? |
| `embedding` | Der eigentliche Float-Vektor. `None` bei sparse-only. |
| `sparse_embedding` | Token-Gewichtungs-Map. `None` bei dense-only. |
| `metadata` | Direkt aus dem Chunk übernommen. Retrieval-Systeme können darüber filtern. |

**Der Vektor allein reicht nicht.** Ein Vektorindex, der nur Float-Arrays speichert,
liefert bei Retrieval eine Distanz, aber keine Antwort auf "Was ist das für ein Text
und woher kommt er?". Die Metadaten schließen diese Lücke.

---

## 10. Embeddings persistieren

Embedding-Berechnungen für alle Chunks einmalig durchzuführen und das Ergebnis
zu speichern ist die zentrale Effizienzentscheidung der Offline-Pipeline. Wenn ein
Nutzer eine Query stellt, soll nicht die gesamte Dokumentenkollektion neu eingebettet
werden, das würde Sekunden bis Minuten dauern.

Der `EmbeddingStore` speichert die Vektoren in einem Append-Only-JSONL-Format:

**Append-Only:** Kein komplexes Update-Protokoll. Ein neuer Lauf schreibt in eine
geleerte Datei.

**Batchweises Schreiben:** Pro Batch öffnet der Store die Datei, schreibt und schließt
wieder. Es gibt keine offene Dateihandle während der gesamten Laufzeit.

**Streamingfähiges Lesen:** `stream_all()` liest Zeile für Zeile, ohne die gesamte
Datei in den RAM zu laden.

In [ ]:
from rag.embedding.store import EmbeddingStore

embedding_store_path = Path("/home/jovyan/workspace/data/embeddings/embeddings.jsonl")

store = EmbeddingStore(path=embedding_store_path)

print(f"Embedding-Store : {embedding_store_path}")
print(f"Verzeichnis erstellt: {embedding_store_path.parent.exists()}")

In [ ]:
# Orchestrator mit Store - persist=True schreibt batchweise
orchestrator_with_store = EmbeddingOrchestrator(
    embedder=embedder,
    config=config,
    store=store,
)

print(f"Persistenz-Lauf mit {len(chunks)} Chunks...")
print()

t0 = time.time()

# stream() mit persist=True:
# 1. store.reset() - leert die Zieldatei
# 2. pro Batch: embed → normalize → store.write_many()
# 3. yield from batch_results - kein RAM-Aufbau
persisted = list(orchestrator_with_store.stream(iter(chunks), persist=True))

elapsed = time.time() - t0

print(f"Embeddings persistiert : {len(persisted)}")
print(f"Zeit                   : {elapsed:.1f}s")
print()
size_kb = embedding_store_path.stat().st_size / 1024
print(f"Dateigroesse             : {size_kb:.1f} KB")
print(f"Pro Embedding (Schaetz) : {(size_kb * 1024 / len(persisted)):.0f} Bytes")

**Warum Persistenz Offline-Arbeit ist:**

Sobald `embeddings.jsonl` existiert und vollständig ist, muss die Embedding-Berechnung
nicht mehr wiederholt werden, bis sich die Chunks oder das Modell ändern. Jede
Nutzer-Query wird zur Laufzeit eingebettet (kurzer Einzel-Forward-Pass), und der
Query-Vektor wird mit den gespeicherten Chunk-Vektoren verglichen.

---

## 11. Integritätsprüfung der Embedding-Persistenz

Eine erfolgreiche Schreiboperation ist keine Garantie für ein korrektes Ergebnis.
Deshalb: Explizite Prüfung nach dem Schreiben.

**Write succeeded ≠ Embeddings korrekt persistiert.** Der nächste Schritt,
der Aufbau des Vektorindex, setzt auf einem korrekten Embedding-Bestand auf.

In [ ]:
# Zurücklesen und prüfen
reloaded_embeddings = store.read_all()

print(f"Embeddings geschrieben : {len(persisted)}")
print(f"Embeddings gelesen     : {len(reloaded_embeddings)}")
print()

if len(reloaded_embeddings) != len(persisted):
    raise RuntimeError(
        f"Integritaetsfehler: {len(persisted)} geschrieben, "
        f"{len(reloaded_embeddings)} gelesen."
    )

print("Anzahl-Check: OK")

In [ ]:
# IDs prüfen: jede chunk_id muss genau einmal in den Embeddings vorkommen
written_chunk_ids = {e["chunk_id"] for e in persisted}
read_chunk_ids    = {e["chunk_id"] for e in reloaded_embeddings}
input_chunk_ids   = {c["id"] for c in chunks}

missing    = input_chunk_ids - read_chunk_ids
unexpected = read_chunk_ids - input_chunk_ids

print("ID-Konsistenz-Check:")
print(f"  Input-Chunks       : {len(input_chunk_ids)}")
print(f"  Embedding-chunk_ids: {len(read_chunk_ids)}")
print(f"  Fehlende           : {len(missing)}")
print(f"  Unerwartete        : {len(unexpected)}")
print()

if missing:
    raise RuntimeError(f"Fehlende chunk_ids: {list(missing)[:5]}")
if unexpected:
    raise RuntimeError(f"Unerwartete chunk_ids: {list(unexpected)[:5]}")

print("ID-Check: OK")

In [ ]:
# Dimensionskonsistenz: alle Dense-Vektoren müssen dieselbe Länge haben
dense_embeddings = [e for e in reloaded_embeddings if e["embedding"] is not None]
dimensions = {len(e["embedding"]) for e in dense_embeddings}

print("Dimensionskonsistenz:")
print(f"  Embeddings mit dense-Vektor : {len(dense_embeddings)}")
print(f"  Distinct Dimensionen        : {dimensions}")
print()

if len(dimensions) > 1:
    raise RuntimeError(
        f"Inkonsistente Dimensionen gefunden: {dimensions}. "
        "Ein Vektorindex kann keine gemischten Dimensionen speichern."
    )

print("Dimensions-Check: OK")
print()

# Modell-Konsistenz
model_types = {e["model_type"] for e in reloaded_embeddings}
print(f"  Modelle im Bestand          : {model_types}")
if len(model_types) > 1:
    print("  WARNUNG: Mehr als ein Modell-Typ! Retrieval-Inkompatibilitaet moeglich.")

**Was dieser Output bedeutet:**

Wenn `Distinct Dimensionen` genau eine Zahl enthält (z.B. `{1024}`), ist der
Embedding-Bestand dimensionskonsistent. Ein Vektorindex akzeptiert nur Vektoren
einheitlicher Dimension.

Wenn mehr als ein Modell-Typ im Bestand existiert, ist das ein Signal für ein Problem:
Entweder wurden zwei Läufe mit verschiedenen Modellen gemischt, oder der Store wurde
nicht korrekt zurückgesetzt.

---

## 12. Bereitschaft für den Vektorindex

Die Embedding Layer endet bei persistierten, geprüften Vektoren. Was diese
abschließende Prüfung feststellt: Ist die Ausgabe der Embedding Layer bereit, als Eingabe
für einen Vektorindex verwendet zu werden?

In [ ]:
n_total = len(reloaded_embeddings)
ids     = [e["id"] for e in reloaded_embeddings]

index_checks = {
    "embeddings.jsonl vorhanden"         : embedding_store_path.exists(),
    "Mindestens 1 Embedding"             : n_total > 0,
    "embedding_id vorhanden"             : all(e["id"] for e in reloaded_embeddings),
    "IDs eindeutig"                      : len(set(ids)) == n_total,
    "chunk_id vorhanden"                 : all(e["chunk_id"] for e in reloaded_embeddings),
    "embedding oder sparse vorhanden"    : all(
        e["embedding"] is not None or e["sparse_embedding"] is not None
        for e in reloaded_embeddings
    ),
    "Dimensionen konsistent"             : len(dimensions) == 1 if dense_embeddings else True,
    "embedding_type gesetzt"             : all(e["embedding_type"] for e in reloaded_embeddings),
    "model_type gesetzt"                 : all(e["model_type"] for e in reloaded_embeddings),
    "metadata vorhanden"                 : all(isinstance(e["metadata"], dict) for e in reloaded_embeddings),
}

all_ready = all(index_checks.values())

print("Vector-Index-Readiness-Check:")
print()
for label, result in index_checks.items():
    status = "OK  " if result else "FAIL"
    print(f"  [{status}]  {label}")

print()
if not all_ready:
    raise RuntimeError("Vector-Index-Readiness-Check fehlgeschlagen - siehe FAIL oben.")

dim = next(iter(dimensions)) if dense_embeddings else "n/a"
print(f"VECTOR-INDEX-READY")
print(f"  {n_total} Embeddings  |  Dimension: {dim}  |  Modell: {next(iter(model_types))}")
print(f"  Store: {embedding_store_path}")

In [ ]:
# Zusammenfassung des Embedding-Bestands
dim = next(iter(dimensions)) if dense_embeddings else "n/a"
print("Embedding-Bestand (Zusammenfassung):")
print()
print(f"  Chunks eingebettet    : {n_total}")
print(f"  Vektordimension       : {dim}")
print(f"  Modell                : {config.model_name}")
print(f"  Modell-Version        : {config.model_version}")
print(f"  Retrieval-Modus       : {config.retrieval_mode}")
print(f"  Normalisiert          : {config.behavior.normalize}")
print(f"  Projektion            : {orchestrator._projection_method}")
print(f"  Store-Pfad            : {embedding_store_path}")
print(f"  Store-Groesse         : {embedding_store_path.stat().st_size / 1024:.1f} KB")

---

## 13. Zusammenfassung: Was die Embedding Layer leistet und was sie bewusst nicht tut

---

### Was hier passiert ist

**Ingestion hat `chunks.jsonl` erzeugt.** Jeder Chunk enthält `id`, `document_id`,
`text` und `metadata`. Die Embedding Layer kennt die Rohdaten nicht. Sie kennt nur
diese Chunks.

**Validation vor dem Modell.** Leere Texte, fehlende Felder, doppelte IDs: all das
wird geprüft, bevor das Modell auch nur initialisiert wird.

**Der BaseEmbedder-Contract.** Der Orchestrator arbeitet gegen eine stabile Schnittstelle,
nicht gegen konkrete Modelle.

**Streaming + Batching.** `batch_iter()` und `stream()` kombinieren Modell-Effizienz
mit RAM-Effizienz.

**Deterministische `embedding_id`.** SHA-256 aus `chunk_id | model_name | model_version`.
Stille Inkonsistenzen zwischen Modellversionen werden verhindert.

**Normalisierung.** L2-Norm auf 1.0. Mathematische Konsistenz für Ähnlichkeitsvergleiche.

**Projektion als explizite Abwägung.** `mrl ≠ truncate`. Die Implementierung ist ehrlich
über den Unterschied.

**Persistenz als Architekturgrenze.** `embeddings.jsonl` ist die Ausgabe der Embedding
Layer und der Input für den Vektorindex-Aufbau.

---

### Was die Embedding Layer bewusst nicht tut

```
Vektorindex-Aufbau        (nächste Stufe: Indexing Layer)
ANN-Strukturen (FAISS, HNSW)
Query-Routing
Retrieval-Scoring
Reranking
Prompt Engineering
LLM-Inferenz
```

---

### Der vollständige Datenfluss bis hierher

```
╔══════════════════════════════════════════════════════════════════╗
║  OFFLINE, abgeschlossen                                          ║
║                                                                  ║
║  Rohdaten                                                        ║
║    → Cleaning (deterministisch)                                  ║
║    → Chunking (struktur-bewusst, mit Fallback)                   ║
║    → chunks.jsonl                                                ║
║                              ← Systemgrenze Ingestion            ║
║    → Validation                                                  ║
║    → BaseEmbedder (embed_documents)                              ║
║    → Projektion (optional)                                       ║
║    → Normalisierung                                              ║
║    → EmbeddingVector (id, chunk_id, embedding, metadata)         ║
║    → embeddings.jsonl                                            ║
║                              ← Systemgrenze Embedding Layer      ║
║    → [nächste Stufe: Indexing Layer]                            ║
╠══════════════════════════════════════════════════════════════════╣
║  ONLINE, noch nicht Teil dieses Notebooks                        ║
║                                                                  ║
║  Query → embed_queries() → Query-Vektor                         ║
║        → Vektorindex-Suche → Top-k Chunks                       ║
║        → LLM → Antwort                                          ║
╚══════════════════════════════════════════════════════════════════╝
```